In [ ]:
import os, sys
from pathlib import Path
_cwd = Path.cwd()
_root = next((p for p in [_cwd] + list(_cwd.parents)
              if (p / 'requirements.txt').exists() and (p / 'src').is_dir()), None)
assert _root, f'Could not find project root from {_cwd}'
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print(f'Project root: {_root}')

# 02 — Covariate Engineering

This notebook demonstrates how the project constructs covariates and enforces the **leakage discipline** that underpins honest backtesting.

Key topics:
1. `build_calendar()` — Fourier + calendar features from a DatetimeIndex
2. Fourier term visualisation (daily sinusoids)
3. Past vs future covariate split — why this matters for backtesting
4. Listing all components from the PanelBundle

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from src.build.fixtures import load_fixture_panel
from src.data.calendar import build_calendar

bundle = load_fixture_panel()
# Recover the raw DatetimeIndex from the target series
ts_index = bundle.target.time_index
print('Index start:', ts_index[0], '  end:', ts_index[-1])
print('Length:', len(ts_index), 'half-hourly steps')

## 1. Build calendar features

In [ ]:
cal = build_calendar(ts_index)
print('Calendar DataFrame shape:', cal.shape)
print('\nFirst 5 rows:')
cal.head()

In [ ]:
print('Calendar columns:')
for col in cal.columns:
    print(f'  {col}')

## 2. Fourier terms — daily sinusoid

The calendar builder produces 2 harmonics for each of daily, weekly, and annual seasonality.  `sin_daily_1` completes one full cycle every 48 SPs (24 hours); `sin_daily_2` completes two cycles.

In [ ]:
# Plot the first 4 days of sin_daily_1 and cos_daily_1
n_plot = 48 * 4  # 4 days
fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)

for i, (ax, col, color) in enumerate(zip(
    axes,
    ['sin_daily_1', 'cos_daily_1'],
    ['royalblue', 'darkorange']
)):
    ax.plot(cal.index[:n_plot], cal[col].values[:n_plot],
            color=color, lw=1.5)
    ax.set_ylabel(col, fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color='k', lw=0.5)

axes[1].set_xlabel('Timestamp (UTC)')
fig.suptitle('Fourier terms — first 4 days', y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# Show sp_of_day cycling 1–48 over a 3-day window
fig, ax = plt.subplots(figsize=(12, 2.5))
n = 48 * 3
ax.plot(cal.index[:n], cal['sp_of_day'].values[:n],
        color='seagreen', lw=1.2, marker='.', markersize=2)
ax.set_ylabel('sp_of_day (1–48)')
ax.set_xlabel('Timestamp (UTC)')
ax.set_title('Settlement period of day — 3-day window')
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## 3. Past vs future covariate split

### The leakage discipline

In a half-hourly electricity market, the model is asked at time **T** to forecast prices for periods **T+1 … T+H**.  Features can only be used at prediction time if they would actually be available at time **T**:

| Covariate type | Available at T? | Used as |
|---|---|---|
| Calendar (dow, sp_of_day, Fourier) | Yes — deterministic | **future** |
| Demand outturn (indo / itsdo) | No — only known after metering | **past** |
| Generation outturn (gen_*) | No — settlement data, not real-time | **past** |

Darts enforces this distinction at model level: past covariates are consumed only up to the origin, future covariates are passed for the entire forecast horizon.

In [ ]:
print('=== bundle.past_covariates ===')
print('Components (', len(bundle.past_covariates.components), '):')
for c in bundle.past_covariates.components:
    print('  ', c)

print()
print('=== bundle.future_covariates ===')
print('Components (', len(bundle.future_covariates.components), '):')
for c in bundle.future_covariates.components:
    print('  ', c)

In [ ]:
# Confirm no calendar columns appear in past_covariates
past_cols  = set(bundle.past_covariates.components.tolist())
fut_cols   = set(bundle.future_covariates.components.tolist())
cal_tokens = {'dow', 'sp_of_day', 'sin_', 'cos_', 'is_weekend', 'is_holiday'}

leakage_candidates = [
    c for c in past_cols
    if any(tok in c for tok in cal_tokens)
]
if leakage_candidates:
    print('WARNING: calendar-like columns in past_covariates:', leakage_candidates)
else:
    print('OK — no calendar columns leaked into past_covariates.')

demand_gen_in_future = [
    c for c in fut_cols
    if c.startswith(('demand_', 'gen_'))
]
if demand_gen_in_future:
    print('WARNING: outturn columns in future_covariates:', demand_gen_in_future)
else:
    print('OK — no outturn columns leaked into future_covariates.')

## 4. Covariate summary statistics

A quick sanity check on the magnitude and variation of each covariate.

In [ ]:
past_df  = bundle.past_covariates.to_dataframe()
fut_df   = bundle.future_covariates.to_dataframe()

print('Past covariates summary:')
print(past_df.describe().T[['mean', 'std', 'min', 'max']].round(1).to_string())
print()
print('Future covariates summary (first 8 cols):')
print(fut_df.iloc[:, :8].describe().T[['mean', 'std', 'min', 'max']].round(3).to_string())